In [1]:
!pip install hospital-sim==0.0.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 102.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 813.6/813.6 kB 22.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 53.4 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-

Laster inn API key fra Open Ai. Legg inn navn og value i "Secrets".

In [2]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY er tom.")

os.environ["OPENAI_API_KEY"] = api_key.strip()
print("API-nøkkelen er lastet inn!")

API-nøkkelen er lastet inn!


Når filen kjører, må det filen fra notebook 1 lastes opp her. Trykk "velg filer"

In [5]:
import json
import pandas as pd
from google.colab import files

# Velg JSON-filen dere lastet ned
uploaded = files.upload()
filnavn = next(iter(uploaded))

# Les inn pasientene
pasienter = json.loads(uploaded[filnavn].decode("utf-8"))

# Kontroller innholdet
print(f"Leste inn {len(pasienter)} pasienter")
display(pd.DataFrame(pasienter).head(10))

Saving pasienter_4_symptomer.json to pasienter_4_symptomer.json
Leste inn 20 pasienter


,profile_id,name,age,gender,chief_complaint,symptoms,medical_history,current_medications,allergies,vital_signs
0,fcf1197f-4c25-4a64-a91b-f4f18408eadc,John Miller,33,Male,Fever,"[Fever, Nausea, Fatigue, Headache]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
1,0e408cc6-a6b4-429a-aeeb-88f1888da86e,Julio Rice,18,Male,Swelling,"[Swelling, Desquamation, Redness, Severe Arm P...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
2,17530cf9-a12b-4832-b6bc-1549f5f46a78,Keith Myers,36,Male,Redness,"[Redness, Breathing Problems, Bruise, Tenderness]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
3,7021e0af-1ffa-4f05-aea3-115ca3e0eeaf,Roger Reveles,52,Male,Tenderness,"[Tenderness, Spasms, Foot Injury, Warmth]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
4,f5169136-961a-4b51-833a-f2249becc835,Debra Givens,49,Female,Red Eye,"[Red Eye, Fatigue, Irritable Cough, Sneezing]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
5,f32f0216-e236-4444-9ea4-163602050637,Hung Rhyner,88,Male,Elbow Pain,"[Elbow Pain, Hand Weakness, Ulnar Claw, Muscle...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
6,a9267ecd-6357-40cb-8ada-d7e1e83dd2d8,Lula Carter,52,Female,Headache,"[Headache, Fever, Dry Cough, Nasal Congestion]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
7,33745826-4326-46cc-8891-b9066c301e17,Rebecca Fanning,52,Female,Swollen Lymph Nodes,"[Swollen Lymph Nodes, Fainting, Fever, Chest P...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
8,9c1c0d28-2b61-4c45-a420-d5221a65d578,Norma Jones,69,Female,Dizziness,"[Dizziness, Hemoptysis, Chest Pain, Breathing ...",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."
9,acfba1ed-fefa-4f0c-ad2c-f76fcbe39fd8,Michael Berka,76,Male,Eschar,"[Eschar, Muscle Pain, Petechiae, Fever]",[],[],[],"{'heart_rate': 80, 'blood_pressure': {'systoli..."


Her kjøres simuleringen vha en modifisert hospital-sim kode, som vi har kalt main

In [6]:
import importlib
import mainhs #laster vår egen fil

importlib.reload(mainhs) #leser filen på nytt, slik at endringer blir brukt

# gjør klassene tilgjengelig under samme navn som før
HospitalSimulation = mainhs.HospitalSimulation
Patient = mainhs.Patient

# Opprett en ny simulering
hospital = HospitalSimulation("Colab Emergency Medical Center")

import inspect

# For å kontrollere hvilken fil som faktisk brukes
print("Sykehuset bruker:", inspect.getfile(type(hospital)))
print(
    "Pulsfunksjonen kommer fra:",
    inspect.getfile(type(hospital)._extract_vital_signs)
)

# Lag pasienter fra alle profilene i JSON-filen
emergency_patients = []

profil_id_per_pasient = {}

for profil in pasienter:
    patient = Patient(
        name=profil["name"],
        age=profil["age"],
        gender=profil["gender"],
        chief_complaint=profil["chief_complaint"],
        symptoms=profil["symptoms"],
        medical_history=profil["medical_history"],
        current_medications=profil["current_medications"],
        allergies=profil["allergies"],
        vital_signs=profil["vital_signs"],
    )

    profil_id_per_pasient[patient.patient_id] = profil["profile_id"]

    emergency_patients.append(patient)

# Legg pasientene inn i sykehuset
for patient in emergency_patients:
    hospital.add_patient(patient)

# Kjør simuleringen til køen er tom
while not hospital.patient_queue.queue.empty():
    hospital.process_patient_queue()

# Vis navn og endelig diagnose for alle pasientene
resultater = pd.DataFrame([
    {
        "Navn": patient.name,
        "Diagnose": patient.diagnosis or "Ingen diagnose registrert",
    }
    for patient in emergency_patients
])

display(resultater)

2026-11-19 07:39:28 | INFO     | mainhs:_initialize_chromadb:268 - EHR System initialized with ChromaDB collection: hospital_ehr
Sykehuset bruker: /content/mainhs.py
Pulsfunksjonen kommer fra: /content/mainhs.py
2026-11-19 07:39:28 | INFO     | mainhs:_initialize_staff:953 - Hospital staff initialized: 8 staff members
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70 - Creating patient: John Miller, Age: 33, Chief Complaint: Fever
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70 - Creating patient: Julio Rice, Age: 18, Chief Complaint: Swelling
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70 - Creating patient: Keith Myers, Age: 36, Chief Complaint: Redness
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70 - Creating patient: Roger Reveles, Age: 52, Chief Complaint: Tenderness
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70 - Creating patient: Debra Givens, Age: 49, Chief Complaint: Red Eye
2026-11-19 07:39:28 | INFO     | mainhs:__post_init__:70

╭────────────────────────────────── Agent Name Crystal Moonwhisper [Loop: 1/1] ───────────────────────────────────╮
│ Good morning! Welcome to our hospital. May I please have your name? Also, could you tell me your age and what   │
│ brings you in today? This will help me get you checked in quickly.                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── Agent Name Patient_John_Miller [Loop: 1/1] ───────────────────────────────────╮
│ Good morning! My name is John Miller, and I’m 33 years old. I’m here because I’ve been having a fever along     │
│ with nausea, feeling really tired, and headaches.                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-11-19 07:39:31 | INFO     | mainhs:add_patient:983 - Patient John Miller checked in and added to queue
2026-11-19 07:39:31 | INFO     | mainhs:add_patient:959 - New patient arrival: Julio Rice - Swelling


╭────────────────────────────────── Agent Name Crystal Moonwhisper [Loop: 1/1] ───────────────────────────────────╮
│ Hello and welcome! Could you please provide me with your full name? I’ll also need your age and a brief         │
│ description of the reason for your visit today. This will help me get you checked in smoothly.                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── Agent Name Patient_Julio_Rice [Loop: 1/1] ───────────────────────────────────╮
│ Good morning! My name is Julio Rice. I'm 18 years old. I'm here because I'm experiencing swelling, redness,     │
│ desquamation, and severe pain in my arm.                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (309 outputs left out)

╭─────────────────────────────── Agent Name Patient_Ethan_Espericueta [Loop: 1/1] ────────────────────────────────╮
│ Hello, Dr. Nightingale. Thank you for seeing me. I'm here because I injured my ankle, and now it's very swollen │
│ and bruised. I'm having a lot of trouble moving it—it's pretty stiff, and the pain gets worse when I try to     │
│ walk or even move it a little. The pain is about a 6 out of 10. I'm worried because I can't put weight on it    │
│ without significant discomfort, and I want to make sure it's not something serious.                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────── Agent Name Dr. Zara Nightingale [Loop: 1/1] ──────────────────────────────────╮
│ Thank you for sharing those details, Ethan. I will carefully examine your ankle now to assess the swelling,     │
│ bruising, tenderness, range of motion, and any signs of instability or fracture. I will also check for pulses   │
│ and sensation to ensure good circulation and nerve function.                                                    │
│                                                                                                                 │
│ Given the injury history, swelling, bruising, stiffness, and pain worsened by movement or weight-bearing, my    │
│ primary concern is a possible ankle sprain with ligament injury or potentially a fracture that needs to be      │
│ ruled out.                                                                                                      │
│                                                                                                                 │
│ I recommend getting an X-ray of your ankle to exclude any bone fractures and to guide appropriate treatment.    │
│ Meanwhile, you should keep the ankle elevated, apply ice to reduce swelling, and avoid putting weight on it. I  │
│ will also consider pain management with acetaminophen or NSAIDs if no contraindications exist.                  │
│                                                                                                                 │
│ If the X-ray shows no fracture, treatment will focus on rest, ice, compression (using an elastic bandage), and  │
│ elevation (RICE protocol), along with physical therapy referral for rehabilitation. If a fracture is detected   │
│ or ligaments are severely damaged, referral to orthopedics is necessary.                                        │
│                                                                                                                 │
│ Please monitor for any increased pain, numbness, color changes, or inability to move toes, which would require  │
│ urgent reevaluation.                                                                                            │
│                                                                                                                 │
│ CLINICAL_SUMMARY_START                                                                                          │
│ DIAGNOSIS: Suspected ankle sprain with possible fracture, diagnosis pending ankle X-ray                         │
│ TREATMENT_PLAN: Order ankle X-ray to exclude fracture; initiate RICE protocol; provide pain management; advise  │
│ non-weight-bearing; arrange orthopedics consult if fracture or severe ligament injury confirmed                 │
│ CLINICAL_SUMMARY_END                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-11-19 08:01:11 | INFO     | mainhs:add_patient_record:286 - Adding patient record to EHR: Ethan Espericueta (ID: da50eac9-857b-435d-ae07-cde11b10c62a) - Doctor: Dr. Zara Nightingale
2026-11-19 08:01:14 | INFO     | mainhs:process_patient_queue:1111 - Patient Ethan Espericueta treated by Dr. Zara Nightingale - Diagnosis: Suspected ankle sprain with possible fracture, diagnosis pending ankle X-ray - Time: 21.8 minutes


,Navn,Diagnose
0,John Miller,Suspected infectious or inflammatory chest con...
1,Julio Rice,Suspected systemic infection or inflammatory c...
2,Keith Myers,Suspected localized soft tissue infection with...
3,Roger Reveles,Suspected soft tissue infection or inflammator...
4,Debra Givens,Likely viral conjunctivitis with associated up...
5,Hung Rhyner,Ulnar nerve entrapment (cubital tunnel syndrom...
6,Lula Carter,Diagnose ikke hentet ut – se samtaleloggen
7,Rebecca Fanning,Suspected systemic illness with lymphadenopath...
8,Norma Jones,"Systemic illness with lymphadenopathy, fever, ..."
9,Michael Berka,Diagnose ikke hentet ut – se samtaleloggen


In [ ]:
from google.colab import drive
drive.mount('/content/drive')